In [1]:
!pip install -q langchain langchain-core langchain-community langchain-huggingface langchain-groq
!pip install -q sentence-transformers faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 16.2 MB/s eta 0:00:00


PDF loading

In [1]:
from google.colab import files

uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]
print(f"✅ Uploaded: {pdf_filename}")

Saving rag_sample_document (1).pdf to rag_sample_document (1).pdf
✅ Uploaded: rag_sample_document (1).pdf


In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from getpass import getpass
import os

load and split document

In [4]:
print("📄 Loading PDF...")
loader = PyPDFLoader(pdf_filename)
pages = loader.load()
print(f"   Total pages: {len(pages)}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = splitter.split_documents(pages)
print(f"   Total chunks: {len(chunks)}")

📄 Loading PDF...
   Total pages: 12
   Total chunks: 39


Embeddings & Vector Store

In [5]:
print("\n🔢 Building embeddings & vector store...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("   ✅ Vector store ready!")


🔢 Building embeddings & vector store...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

   ✅ Vector store ready!


Groq LLM

In [6]:
print("\n🤖 Setting up Groq LLM...")
os.environ["GROQ_API_KEY"] = getpass("   Enter your Groq API key: ")
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.3,
    max_tokens=512
)
print("   ✅ Groq LLM ready!")


🤖 Setting up Groq LLM...
   Enter your Groq API key: ··········
   ✅ Groq LLM ready!


Prompt Template

In [7]:
prompt_template = """Use the following context from the document to answer the question.
If the answer is not in the context, say "I don't know based on the provided document."

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

RAG Chain

In [8]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("\n✅ RAG chain ready! Run the next cell to start asking questions.")


✅ RAG chain ready! Run the next cell to start asking questions.


Q&A Loop

In [9]:
print("=" * 55)
print("📄  RAG App — Ask questions about your PDF")
print("    Type 'quit' to exit")
print("=" * 55)

while True:
    question = input("\nYour question: ").strip()

    if question.lower() in ["quit", "exit", "q"]:
        print("Goodbye!")
        break

    if not question:
        continue

    # Get answer
    answer = qa_chain.invoke(question)
    print("\n💡 Answer:", answer)

    # Show source chunks
    print("\n📌 Sources:")
    for doc in retriever.invoke(question):
        page = doc.metadata.get("page", "?")
        print(f"   - Page {page + 1}: {doc.page_content[:80]}...")

    print("-" * 55)


📄  RAG App — Ask questions about your PDF
    Type 'quit' to exit

Your question: What is this document about?

💡 Answer: This document is about Artificial Intelligence in Healthcare, specifically serving as a comprehensive reference guide for testing Retrieval-Augmented Generation (RAG) pipelines, covering diverse topics within healthcare AI and machine learning.

📌 Sources:
   - Page 7: Chapter 4: Clinical Decision Support Systems
4.1 Electronic Health Records and N...
   - Page 1: Artificial Intelligence in Healthcare
 A Comprehensive Reference Guide for RAG T...
   - Page 10: NLP (Natural Language Processing): AI techniques for understanding and generatin...
-------------------------------------------------------

Your question: What are the main topics covered in this document?

💡 Answer: Based on the provided context, the main topics covered in this document appear to be:

1. Clinical Decision Support Systems
2. Electronic Health Records and Natural Language Processing (NLP)
3. R